In [1]:
import pandas as pd

def calculate_engagement_metrics(df):
    # --- Basic Engagements ---
    df["total_engagements"] = df[["likes", "replies", "reposts", "quotes", "bookmarks"]].sum(axis=1)

    # --- Engagement Ratios (per post, normalized by views) ---
    df["engagement_rate"] = df["total_engagements"] / df["views"] * 100
    df["like_rate"]       = df["likes"]   / df["views"] * 100
    df["reply_rate"]      = df["replies"] / df["views"] * 100
    df["repost_rate"]     = df["reposts"] / df["views"] * 100
    df["quote_rate"]      = df["quotes"]  / df["views"] * 100
    df["bookmark_rate"]   = df["bookmarks"] / df["views"] * 100

    # Handle division by zero safely (replace inf/nan with 0)
    df.replace([float("inf"), float("-inf")], 0, inplace=True)
    df.fillna(0, inplace=True)

    # --- Profile-Level Metrics (aggregate) ---
    profile_metrics = {}
    
    # Total sums
    profile_metrics["total_likes"] = df["likes"].sum()
    profile_metrics["total_replies"] = df["replies"].sum()
    profile_metrics["total_reposts"] = df["reposts"].sum()
    profile_metrics["total_quotes"] = df["quotes"].sum()
    profile_metrics["total_bookmarks"] = df["bookmarks"].sum()
    profile_metrics["total_engagements"] = df["total_engagements"].sum()
    profile_metrics["total_views"] = df["views"].sum()

    # Engagement per follower (requires followers column)
    if "followers" in df.columns:
        avg_followers = df["followers"].mean()  # if followers is per-row, take avg
        profile_metrics["engagement_per_follower"] = (
            profile_metrics["total_engagements"] / avg_followers
            if avg_followers > 0 else 0
        )

    # Average engagement rate across posts
    profile_metrics["avg_engagement_rate"] = df["engagement_rate"].mean()

    # Posting frequency (needs account age if available)
    if "posts_count" in df.columns and "date_posted" in df.columns:
        df["date_posted"] = pd.to_datetime(df["date_posted"], errors="coerce")
        account_age_days = (pd.Timestamp.now() - df["date_posted"].min()).days
        profile_metrics["posting_frequency"] = (
            df["posts_count"].max() / account_age_days
            if account_age_days > 0 else 0
        )

    return df, profile_metrics


# --- Example usage ---
df = pd.read_csv("twitter-posts.csv")

df_metrics, profile_summary = calculate_engagement_metrics(df)

print("\n--- Per Post Engagement Metrics ---")
print(df_metrics[[
    "id", "description", "likes", "replies", "reposts", "quotes", "bookmarks", 
    "views", "total_engagements", "engagement_rate", "like_rate", "reply_rate", 
    "repost_rate", "quote_rate", "bookmark_rate"
]].head())

print("\n--- Profile Level Summary ---")
for k, v in profile_summary.items():
    print(f"{k}: {v}")



--- Per Post Engagement Metrics ---
                    id                                        description  \
0  1795704262074507432  What is he gonna do about it though? Jumped on...   
1  1720922514690375706  In a place where every drop of water counts, I...   
2  1727113960967885140                                May he rest in piss   
3  1800863719293083764  i think it’s time we trend the tags again! HYB...   
4  1799608510902370599  The same question must be asked of @PrideToron...   

   likes  replies  reposts  quotes  bookmarks     views  total_engagements  \
0    699       11       27       1          6   30879.0                744   
1   1585       42     1546     127         61  146375.0               3361   
2   1385       38       71       0         33   97431.0               1527   
3   2073      197     2130      24         59   33867.0               4483   
4    141        8       50       1          5    7779.0                205   

   engagement_rate  like_rate  

C:\Users\theay\AppData\Local\Temp\ipykernel_19988\1123813140.py:44: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["date_posted"] = pd.to_datetime(df["date_posted"], errors="coerce")
